# LaflaAi-Core 400M 4000 Step Tam Egitim

Bu defter smoke/test egitimi degildir. Repo modullerini cagirir, Drive'a final checkpoint arsivi yazar ve shell komutlarini Python hucrelerinde `!` ile calistirir.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
from pathlib import Path
repo = Path('/content/LaflaAi-Core')
assert repo.exists(), 'LaflaAi-Core klasoru /content/LaflaAi-Core altinda olmali.'
%cd /content/LaflaAi-Core

In [ ]:
!python -m pip install -r requirements/colab.txt

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.check_environment --optimizer adamw8bit
!PYTHONPATH=src python -m lafla_ai_core.cli.quality_scan --root .
!PYTHONPATH=src python -m lafla_ai_core.cli.preflight configs/model/lafla-400m-thinking.yaml configs/training/colab-t4-400m-4000.yaml configs/tokenizer/turkish-thinking-bpe.yaml configs/runtime/desktop-cpu-4bit.yaml configs/post_training/lafla-thinking-sft.yaml

In [ ]:
!mkdir -p /content/LaflaAI/tokenizer /content/LaflaAI/reports /content/LaflaAI/checkpoints
!PYTHONPATH=src python -m lafla_ai_core.cli.data_audit --manifest /content/LaflaAI/data/veri_manifesti.json --report /content/LaflaAI/reports/data-audit.json
!PYTHONPATH=src python -m lafla_ai_core.cli.tokenizer_train --config configs/tokenizer/turkish-thinking-bpe.yaml --output /content/LaflaAI/tokenizer/lafla-tokenizer.json --report /content/LaflaAI/reports/tokenizer-quality.json configs/data/lafla-model-identity.jsonl /content/LaflaAI/data/train.jsonl

In [ ]:
!PYTHONPATH=src python -m lafla_ai_core.cli.train_pretrain --model-config configs/model/lafla-400m-thinking.yaml --training-config configs/training/colab-t4-400m-4000.yaml --tokenizer-path /content/LaflaAI/tokenizer/lafla-tokenizer.json --checkpoint-dir /content/LaflaAI/checkpoints --health-log /content/LaflaAI/reports/train-health.jsonl --data-jsonl configs/data/lafla-model-identity.jsonl --data-jsonl /content/LaflaAI/data/train.jsonl

In [ ]:
!mkdir -p /content/gdrive/MyDrive/LaflaAI/final-checkpoint
!tar -czf /content/gdrive/MyDrive/LaflaAI/final-checkpoint/lafla-400m-thinking-4000-step-run.tar.gz -C /content/LaflaAI/checkpoints lafla-final -C /content/LaflaAI tokenizer reports
!sync && ls -lh /content/gdrive/MyDrive/LaflaAI/final-checkpoint